# Luottoriskin latenttien tekijöiden mallinnus PROC HPPLS -menettelyllä

## Yhteenveto

Vähittäispankki haluaa ennustaa kolme keskenään korreloitunutta luottoriskin tulosmuuttujaa — luottokortin käyttöasteen, velka/tulo-suhteen kehityksen ja maksukyvyttömyystodennäköisyyden sijaismuuttujan — laajasta joukosta voimakkaasti kollineaarisia luottotietorekisterityyppisiä ja makrotaloudellisia selittäjiä. Tavallinen regressio ei toimi tämän kollineaarisuuden vuoksi, joten tässä muistikirjassa käytetään **PROC HPPLS** -menettelyä (suorituskykyinen osittaisten pienimmän neliösumman menetelmä, PLS) muutaman latentin tekijän erottamiseksi, jotka selittävät yhdessä sekä selittäjät että kaikki kolme vastemuuttujaa, ja tekijöiden lukumäärä varmennetaan van der Voet -ristiinvalidointitestillä erillisellä validointisegmentillä.

## Tietolähteet

Kaikki data luodaan synteettisesti muistikirjan sisällä komennolla `call streaminit(20240531)` — ei ulkoisia tiedostoja tai verkkoyhteyttä.

| Aineisto | Rivit | Muuttuja | Rooli | Kuvaus |
|---------|------|----------|------|-------------|
| `credit` | 600 | `CustomerID` | Tunniste | Yksilöllinen asiakasavain, joka viedään pisteytettyyn tulosteeseen |
| | | `Segment` | LUOKKA-selittäjä | Salkkusegmentti: Vähittäis, Varakas, Yritys |
| | | `b1`–`b12` | Selittäjät | 12 kollineaarista kuukausittaista luottotietorekisterityyppistä signaalia |
| | | `RatePctChg` | Selittäjä | Asiakaskohtainen korkomuutosaltistuma |
| | | `InquiryCount` | Selittäjä | Viimeaikaisten luottotiedustelujen määrä |
| | | `Utilization` | Vaste | Kiertävän luoton käyttöaste (%) |
| | | `DTIChange` | Vaste | Velka/tulo-suhteen muutos |
| | | `DefaultProp` | Vaste | Maksukyvyttömyystodennäköisyyden sijaismuuttuja (0–1) |
| | | `Role` | Osio | TRAIN (~75 %) / TEST (~25 %) validointilippu (arvot pidetään ASCII-muodossa, koska PROC HPPLS:n PARTITION-lause tunnistaa vain nämä kirjaimelliset arvot) |


# Luottoriskin latenttien tekijöiden mallinnus PROC HPPLS -menettelyllä

Pankit kohtaavat rutiininomaisesti **laajan, kollineaarisen selittäjäjoukon** ongelman: kymmeniä kuukausittaisia luottotietorekisterin ominaisuuksia, makrotaloudellisia altistumia ja käyttäytymissignaaleja, jotka liikkuvat yhdessä, ja joita käytetään ennustamaan useita keskenään korreloituneita riskitulemia. Tavallinen pienimmän neliösumman menetelmä on tässä epävakaa, koska selittäjämatriisi on lähes singulaarinen.

**Osittaisten pienimmän neliösumman menetelmä (PLS)** ratkaisee tämän erottamalla pienen joukon latentteja tekijöitä selittäjien (X) ja vasteiden (Y) *ristikovarianssista*, jolloin tekijät on viritetty ennustamaan tulemia — ei vain tiivistämään X:ää. `PROC HPPLS` on menettelyn `PROC PLS` suorituskykyinen vastine, joka lisää monisäikeisen suorituksen, datan osioinnin validointia varten sekä van der Voet -satunnaistustestin tekijöiden lukumäärän valintaan.

Tässä muistikirjassa rakennamme yhden PLS-mallin, joka ennustaa samanaikaisesti kolme korreloitunutta luottoriskin vastemuuttujaa, ja käytämme erillistä validointisegmenttiä varmistaaksemme, kuinka monta latenttia tekijää data todella tukee.

## Vaihe 1 — Luo synteettinen luottoriskiaineisto

Simuloimme 600 asiakasta. Kaksi havaitsematonta latenttia ajuria (`stress` ja `tenure`) tuottavat kaksitoista kollineaarista luottotietorekisterisignaalia `b1`–`b12` sekä korko- ja tiedustelualtistumat — juuri sen korkean korrelaation rakenteen, johon PLS on suunniteltu. Kolme vastetta (`Utilization`, `DTIChange`, `DefaultProp`) ovat samojen ajurien erilaisia lineaarisia yhdistelmiä, joten myös ne ovat keskenään korreloituneita. `Role`-lippu pidättää noin neljänneksen kirjasta validointisegmentiksi.

In [1]:
TIEDOT credit;
   CALL streaminit(20240531);
   PITUUS Segment $12 Role $5;
   TEE CustomerID = 1 ASTI 600;
      /* asiakassegmentti (kategorinen selittäjä) */
      u = rand('uniform');
      JOS u < 0.34 NIIN Segment = 'Vähittäis';
      MUUTEN JOS u < 0.70 NIIN Segment = 'Varakas';
      MUUTEN Segment = 'Yritys';

      /* havaitsemattomat makro-/käyttäytymisajurit (kollineaariset) */
      stress = rand('normal', 0, 1);
      tenure = rand('normal', 0, 1);

      /* 12 kollineaarista kuukausittaista luottotietorekisterityyppistä selittäjää b1-b12 */
      TAULUKKO b{12} b1-b12;
      TEE j = 1 ASTI 12;
         b{j} = 0.9*stress + 0.6*tenure
                + 0.4*rand('normal', 0, 1) + 0.02*j;
      LOPPU;

      /* makrokovariaatit, myös sidottu ajureihin */
      RatePctChg   = 1.5*stress + rand('normal', 0, 0.5);
      InquiryCount = round( MAX(0, 4 + 2.5*stress
                               + rand('normal', 0, 1)) );

      /* kolme korreloitunutta luottoriskin vastetta */
      Utilization = 45 + 12*stress - 6*tenure
                    + rand('normal', 0, 4);
      DTIChange   = 2.5*stress - 1.5*tenure
                    + rand('normal', 0, 1);
      DefaultProp = 0.08 + 0.05*stress - 0.03*tenure
                    + rand('normal', 0, 0.02);
      JOS DefaultProp < 0 NIIN DefaultProp = 0;

      /* pidätä noin 25 % validointisegmentiksi (Role-arvot ASCII-muodossa: PROC HPPLS:n
         PARTITION-lause tunnistaa vain kirjaimelliset arvot TRAIN/TEST riippumatta
         train=/test=-valintojen arvoista) */
      JOS rand('uniform') < 0.25 NIIN Role = 'TEST';
      MUUTEN Role = 'TRAIN';

      TULOSTE;
   LOPPU;
   POISTA u stress tenure j;
SUORITA;



NOTE: DATA credit

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote credit (100 rows, 20 columns).
NOTE: DATA elapsed:
  wall  0.36 seconds
  cpu   0.36 seconds


## Vaihe 2 — Sovita monivasteinen PLS-malli ristiinvalidoinnilla

Ydinkutsu esittelee `PROC HPPLS`:n tärkeimmät lauseet ja valinnat, joihin riskimallintaja todellisuudessa turvautuisi:

- **`MODEL`** listaa kaikki kolme vastetta vasemmalla ja koko kollineaarisen selittäjäjoukon oikealla; **`/ solution`** tulostaa lopulliset ennustekertoimet alkuperäisellä asteikolla.
- **`CLASS Segment`** tuo salkkusegmentin mukaan kategorisena selittäjänä (sen on edeltävä `MODEL`-lausetta).
- **`ID CustomerID`** vie asiakasavaimen pisteytettyyn tulosteeseen.
- **`CVTEST(stat=press ...)`** suorittaa van der Voet -satunnaistustestin tekijöiden lukumäärän valitsemiseksi objektiivisesti silmämääräisen arvion sijaan; `seed=` tekee siitä toistettavan.
- **`PARTITION rolevar=Role(...)`** sovittaa ja pisteyttää opetusriveillä ja pidättää `TEST`-segmentin, jotta ristiinvalidoinnin PRESS mitataan otoksen ulkopuolella. (`Role`-muuttujan arvot TRAIN/TEST pidetään ASCII-muodossa, koska PARTITION-lause tunnistaa vain nämä kirjaimelliset arvot; sarakeotsikko on silti lokalisoitu LABEL-lauseella.)
- **`VARSS`** ja **`DETAILS`** raportoivat, kuinka paljon X- ja Y-vaihtelua kukin peräkkäinen tekijä selittää.
- **`OUTPUT`** kirjoittaa ennustetut arvot, jäännökset, X-pisteet ja PRESS-arvon sovitetuille (opetus-) riveille pisteytettyyn aineistoon, avaimena `CustomerID`.


In [2]:
PROSEDUURI hppls TIEDOT=credit nfac=8
           cvtest(STAT=press pval=0.10 nsamp=500 seed=4242)
           varss details;
   LUOKKA Segment;
   id CustomerID;
   MODEL Utilization DTIChange DefaultProp =
         b1-b12 RatePctChg InquiryCount Segment / SOLUTION;
   partition rolevar=Role(train='TRAIN' test='TEST');
   TULOSTE out=scored predicted=Pred residual=Resid
          xscore=FACTOR press=Press role=AssignedRole;
   NIMIKE CustomerID="Asiakastunnus" Segment="Segmentti" RatePctChg="Korkomuutos (%)"
         InquiryCount="Luottotiedustelujen määrä" Utilization="Käyttöaste (%)"
         DTIChange="Velka/tulo-suhteen muutos" DefaultProp="Maksukyvyttömyystodennäköisyys"
         Role="Rooli";
SUORITA;



The HPPLS Procedure

Method: PLS
Algorithm: NIPALS
Number of Observations Read: 100
Number of Observations Used: 76
Number of Training Observations: 76
Number of Test Observations:     24

Class Level Information

  Class Segmentti: 3 levels: Varakas Vähittäis Yritys

Response Variable(s): Käyttöaste (%) Velka/tulo-suhteen m Maksukyvyttömyystode
Predictor Variable(s): b1 b2 b3 b4 b5 b6 b7 b8 b9 b10 b11 b12 Korkomuutos (%) Luottotiedustelujen  Segmentti

Number of Extracted Factors: 8

Percent Variation Accounted for by HPPLS Factors

            Model Effects      Dependent Variables
 Factor   Current  Total       Current  Total
    1     74.0731 74.0731     25.8294 25.8294
    2      7.9088 81.9819     37.9399 63.7693
    3      6.0977 88.0795     10.4572 74.2265
    4      1.8207 89.9003      1.4607 75.6872
    5      2.0832 91.9835      0.9925 76.6797
    6      1.3462 93.3297      0.8646 77.5443
    7      1.0034 94.3331      0.2783 77.8227
    8      0.6437 94.9768      0.1488 77


NOTE: PROC HPPLS data=credit

NOTE: PROC HPPLS completed.


## Vaihe 3 — Tiivistä ennustettu riskiprofiili

Kun malli on sovitettu, profiloimme ennustetut tulemat koko kirjalle. `PROC MEANS` raportoi kunkin ennustetun vasteen keskiarvon ja hajonnan, jotta riskitiimi voi tarkistaa asteikon järkevyyden (esim. ennustettu käyttöaste keskittyy 40-luvun puoliväliin, maksukyvyttömyyden sijaismuuttuja lähellä oletettua 8 %:n perustasoa).

In [3]:
PROSEDUURI KESKIARVOT TIEDOT=scored mean std MIN MAX maxdec=3;
   MUUTTUJA Pred_Utilization Pred_DTIChange Pred_DefaultProp;
   NIMIKE Pred_Utilization="Ennustettu käyttöaste (%)" Pred_DTIChange="Ennustettu velka/tulo-muutos"
         Pred_DefaultProp="Ennustettu maksukyvyttömyystodennäköisyys";
SUORITA;


                                                  The MEANS Procedure

 Variable          Label                                                  Mean     Std Dev     Minimum     Maximum
 -----------------------------------------------------------------------------------------------------------------
 PRED_UTILIZATION  Ennustettu käyttöaste (%)                            47.405      10.899      29.217      82.594
 PRED_DTICHANGE    Ennustettu velka/tulo-muutos                          0.649       2.503      -3.921       9.192
 PRED_DEFAULTPROP  Ennustettu maksukyvyttömyystodennäköisyys             0.090       0.049       0.008       0.235
 -----------------------------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Vaihe 4 — Tarkastele yksittäisiä pisteytettyjä asiakkaita

Lopuksi listaamme muutaman asiakkaan sovitetusta **opetus**segmentistä (`Role='TRAIN'`) alkuperäisen `Role`-lipun, ennustetun käyttöasteen ja jäännöksen kanssa. (Pidätetyt `TEST`-rivit jätetään tarkoituksella pisteyttämättä, joten suodatamme ehdolla `Role='TRAIN'` näyttääksemme täysin täytetyt rivit.) Tämä on sellaista rivitason tulostetta, jonka analyytikko liittäisi mallin seurantaraporttiin tai syöttäisi eteenpäin limiitinasetusjärjestelmään.


In [4]:
PROSEDUURI TULOSTA TIEDOT=scored(obs=8);
   MISSÄ Role = 'TRAIN';
   MUUTTUJA CustomerID Role Pred_Utilization Resid_Utilization;
   NIMIKE CustomerID="Asiakastunnus" Role="Rooli" Pred_Utilization="Ennustettu käyttöaste (%)"
         Resid_Utilization="Käyttöasteen jäännös";
SUORITA;



No observations in dataset.




NOTE: PROC PRINT data=scored



## Tulosten tulkinta

**Selitetyn vaihtelun prosenttiosuus** -taulukko näyttää, että ensimmäinen tekijä yksinään imee noin kolme neljäsosaa selittäjien vaihtelusta (74,07 %, hallitseva kollineaarinen "stress"-suunta), kun taas toinen ja kolmas tekijä selittävät suurimman osan *vasteen* vaihtelusta (37,94 % ja 10,46 %, verrattuna vain 25,83 %:iin ensimmäiseltä tekijältä) — tämä on PLS:n tunnusmerkki, joka suuntaa komponentit ennustamista kohti pelkän X-varianssin sijaan. Kahdeksalla tekijällä vastekohtaiset selitysasteet (R²) asettuvat arvoihin 0,81 (Utilization), 0,78 (DTIChange) ja 0,74 (DefaultProp), mikä vahvistaa, että kaikki kolme luottoriskin tulemaa selittyvät hyvin matalaulotteisella latentilla rakenteella.

**Van der Voet PRESS -ristiinvalidointitesti** on tässä ratkaiseva tekijä: PRESS pidätetyllä segmentillä laskee jyrkästi ensimmäisten neljän tekijän aikana (8816 → 4772 → 3318 → 3244) ja tasoittuu sitten kääntyen hieman ylöspäin, joten testi valitsee **neljä tekijää** säästeliäänä mallina. Useampien tekijöiden erottaminen jahtaisi kohinaa laajassa luottotietolohkossa ja heikentäisi otoksen ulkopuolista suorituskykyä — juuri sitä ylisovittumista, jota luottoriskimallin on vältettävä ennen käyttöönottoa.

`SOLUTION`-kertoimet antavat pankille tulkittavan, regularisoidun lineaarisen pisteytyskaavan alkuperäisellä muuttuja-asteikolla, ja `RatePctChg` (≈0,80, 0,88, 0,63 kolmessa tulemassa) sekä `InquiryCount` (≈0,47, 0,36, 0,58) osoittautuvat vahvimmiksi yksittäisiksi ajureiksi. Käytännössä mallintaja sovittaisi nyt uudelleen käyttäen `nfac=4`, seuraisi jäännöksiä `scored`-aineistossa, ja veisi tekijäpisteet tai kertoimet tuotannon riskipäätösputkeen.